## Google colab GPU

In [22]:
# Colab: clone repo + add to path
import os
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/colinminini/LLM6G-Repository'
REPO_DIR = 'LLM6G-Repository'

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.check_call(['git', 'clone', REPO_URL])
    os.chdir(REPO_DIR)
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())
else:
    print('Repo clone skipped (not running in Colab).')

HAS_CUDA = False
try:
    import torch
    HAS_CUDA = torch.cuda.is_available()
except Exception:
    pass

if IN_COLAB and HAS_CUDA:
    # Install dependencies only when running on Colab with GPU
    !pip -q install -U pip
    !pip -q install -r requirements.txt
    # Optional: verify GPU
    !nvidia-smi
else:
    print(f'GPU setup skipped (colab={IN_COLAB}, cuda={HAS_CUDA}).')


Repo clone skipped (not running in Colab).
GPU setup skipped (colab=False, cuda=False).


# Model Benchmarking (Coverage)

Train LSTM, DeepAR, and TFT on `data_1to7`, then fine-tune Chronos-2 on the same dataset. The summary reports one-sided q0.95 coverage on train/val/test and uses quantile training (q50, q95).


In [23]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def _resolve_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError("Could not locate project root containing 'src/' and 'data/'.")


PROJECT_ROOT = _resolve_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from src.train_utils import train_all
from src.chronos2_finetune import (
    Chronos2FineTuneConfig,
    build_default_hyperparameters,
    load_splits_for_autogluon,
    run_chronos2_finetune,
    evaluate_chronos2_models,
)

MODEL_TYPES = ["lstm", "deepar"] # add , 'tft' if wanted
DATASET = "data_1to7"
SPLITS = ["train", "val", "test"]

CONTEXT_LENGTH = 48
FORECAST_LENGTH = 48
QUANTILES = (0.5, 0.95)
MAX_EPOCHS = 100
PATIENCE = 3
HIDDEN_SIZE = 128
NUM_LAYERS = 2
NUM_HEADS = 4
COVERAGE_TARGET = 0.95 if 0.95 in QUANTILES else max(QUANTILES)

AUTOGLUON_TIME_LIMIT = 3000  # seconds
AUTOGLUON_EVAL_METRIC = "MAE"
AUTOGLUON_FREQ = "s"  # synthetic regular cadence
AUTOGLUON_KNOWN_COVARIATES = []

RESULTS_DIR = PROJECT_ROOT / "results/benchmarks"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CHRONOS2_CONFIG = Chronos2FineTuneConfig(
    dataset_base=DATASET,
    prediction_length=FORECAST_LENGTH,
    quantiles=QUANTILES,
    freq=AUTOGLUON_FREQ,
    use_integer_timestamps=True,
    synthetic_start="2000-01-01 00:00:00",
    eval_metric=AUTOGLUON_EVAL_METRIC,
    time_limit=AUTOGLUON_TIME_LIMIT,
    known_covariates_names=AUTOGLUON_KNOWN_COVARIATES,
    results_dir=RESULTS_DIR,
    hyperparameters=build_default_hyperparameters(),
    # hyperparameters={"Chronos2": [
    #        {"ag_args": {"name_suffix": "ZeroShot"},
    #         "context_length": CONTEXT_LENGTH},
    #        {"fine_tune": True, "ag_args": {"name_suffix": "FineTuned"},
    #         "context_length": CONTEXT_LENGTH},
    #    ]},

)


## Train baseline models

This runs LSTM, DeepAR, and TFT on `data_1to7`. Coverage shown is the final epoch value returned by the trainer.


In [24]:
train_results, train_histories = train_all(
    MODEL_TYPES,
    DATASET,
    context_length=CONTEXT_LENGTH,
    forecast_length=FORECAST_LENGTH,
    quantiles=QUANTILES,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
)

train_rows = []
for (dataset, model), metrics in train_results.items():
    for split, coverage in metrics.items():
        train_rows.append(
            {"dataset": dataset, "split": split, "model": model, "coverage": coverage}
        )

train_df = pd.DataFrame(train_rows)
train_pivot = train_df.pivot_table(
    index=["dataset", "split"], columns="model", values="coverage"
).sort_index()
train_pivot


Epoch 1/100 - train_loss=0.623 - train_rmse=3.634 - val_loss=0.921 - val_rmse=5.658 - test_loss=1.066 - test_rmse=6.521 
Epoch 2/100 - train_loss=0.538 - train_rmse=3.574 - val_loss=0.909 - val_rmse=5.396 - test_loss=1.030 - test_rmse=6.278 
Epoch 3/100 - train_loss=0.520 - train_rmse=3.225 - val_loss=0.843 - val_rmse=4.618 - test_loss=0.962 - test_rmse=5.431 
Early stopping at epoch 4: train_rmse=3.030, val_rmse=4.554, test_rmse=5.389


'Final metrics'

,metric,value
0,train_loss,0.501
1,train_rmse,3.030
2,val_loss,0.825
3,val_rmse,4.554
4,train_q0.5,0.799
5,train_q0.95,0.204
6,val_q0.5,1.321
7,val_q0.95,0.332
8,train_coverage,0.949
9,val_coverage,0.926


Epoch 1/100 - train_loss=1.682 - train_rmse=2.554 - val_loss=2.123 - val_rmse=3.803 - test_loss=2.152 - test_rmse=4.239 
Epoch 2/100 - train_loss=1.567 - train_rmse=2.411 - val_loss=2.107 - val_rmse=3.794 - test_loss=2.137 - test_rmse=4.235 
Epoch 3/100 - train_loss=1.552 - train_rmse=2.399 - val_loss=2.095 - val_rmse=3.784 - test_loss=2.119 - test_rmse=4.204 
Early stopping at epoch 4: train_rmse=2.381, val_rmse=3.782, test_rmse=4.196


'Final metrics'

,metric,value
0,train_loss,1.529
1,train_rmse,2.381
2,val_loss,2.097
3,val_rmse,3.782
4,train_q0.5,0.757
5,train_q0.95,0.171
6,val_q0.5,1.274
7,val_q0.95,0.282
8,train_coverage,0.947
9,val_coverage,0.957


model              deepar      lstm
dataset   split                    
data_1to7 test   0.954736  0.903222
          train  0.946618  0.949249
          val    0.957437  0.926429

## AutoGluon Chronos-2 fine-tuning

We fine-tune Chronos-2 using AutoGluon TimeSeries with a zero-shot and fine-tuned run.


In [ ]:
train_data, val_data, test_data, data_freq = load_splits_for_autogluon(CHRONOS2_CONFIG)

predictor, model_names, leaderboard = run_chronos2_finetune(
    CHRONOS2_CONFIG,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    freq=data_freq,
)

autogluon_df = evaluate_chronos2_models(
    predictor,
    model_names=model_names,
    splits={"train": train_data, "val": val_data, "test": test_data},
    dataset_label=DATASET,
    prediction_length=FORECAST_LENGTH,
    quantiles=QUANTILES,
)

autogluon_coverage_pivot = autogluon_df.pivot_table(
    index=["dataset", "split"], columns="model", values="coverage"
).sort_index()
autogluon_mae_pivot = autogluon_df.pivot_table(
    index=["dataset", "split"], columns="model", values="mae"
).sort_index()

autogluon_coverage_pivot, autogluon_mae_pivot, leaderboard


## Coverage and MAE comparison summary

This table and plot compare one-sided q0.95 coverage across train/val/test for all models.
The AutoGluon tables above include both coverage and MAE for Chronos-2
ZeroShot vs FineTuned.


In [ ]:
summary_df = pd.concat(
    [train_df, autogluon_df[["dataset", "split", "model", "coverage"]]],
    ignore_index=True,
)
summary_df["dataset_label"] = summary_df["dataset"].str.replace("data_", "") + "/" + summary_df["split"]
order = [
    "1to7/train",
    "1to7/val",
    "1to7/test",
]

summary_pivot = summary_df.pivot_table(
    index="dataset_label", columns="model", values="coverage"
).reindex(order)
summary_pivot  # type: ignore

ax = summary_pivot.plot(kind="bar", figsize=(12, 6))
ax.axhline(COVERAGE_TARGET, color="black", linestyle="--", linewidth=1, label=f"{COVERAGE_TARGET:.2f} target")
ax.set_ylabel("Coverage")
ax.set_title("One-sided q0.95 coverage across train/val/test")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=45, ha="right")
ax.legend()
plt.tight_layout()
PLOTS_DIR = PROJECT_ROOT / "results/plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
benchmark_plot_path = PLOTS_DIR / "benchmark_train_models_display_coverage.png"
plt.savefig(benchmark_plot_path, dpi=160, bbox_inches="tight")
plt.show()
print("Saved plot:", benchmark_plot_path)

autogluon_mae_pivot
